In [1]:
import pandas as pd
import numpy as np

MAPPING = "../Progress/code/datasets/mapping"

# Load pre-computed files from final_v2
best_match      = pd.read_csv(f"{MAPPING}/program_career_sentence_transformer_best_thr30.csv")
occupation_gcsi = pd.read_csv(f"{MAPPING}/occupation_gcsi.csv")
skill_bridge    = pd.read_csv(f"{MAPPING}/occupation_skill_bridge_onet.csv", low_memory=False)
tech_summary    = pd.read_csv(f"{MAPPING}/occupation_technology_skills_onet_summary.csv")

print("Loaded.")
print(f"  best_match:      {best_match.shape}")
print(f"  occupation_gcsi: {occupation_gcsi.shape}")

Loaded.
  best_match:      (57085, 8)
  occupation_gcsi: (702, 11)


In [2]:
import re

full_map = {
    "biological scientists":                              "biological scientists; all other",
    "psychologists":                                      "clinical; counseling; and school psychologists",
    "education administrators":                           "education administrators; all other",
    "education administrators postsecondary":             "education administrators; postsecondary",
    "foreign language and literature teachers postsecondary": "postsecondary teachers; all other",
    "legal occupations":                                  "lawyers",
    "english language and literature teachers postsecondary": "postsecondary teachers; all other",
    "engineers":                                          "engineers; all other",
    "media and communication workers":                    "public relations specialists",
    "media and communication workers all other":          "public relations specialists",
    "business operations specialists":                    "business operations specialists; all other",
    "business operations specialists all other":          "business operations specialists; all other",
    "community health workers":                           "health educators",
    "management occupations":                             "general and operations managers",
    "medical scientists":                                 "medical scientists; except epidemiologists",
    "art drama and music teachers postsecondary":         "postsecondary teachers; all other",
    "social workers":                                     "social workers; all other",
    "electrical and electronics engineers":               "electrical engineers",
    "philosophy and religion teachers postsecondary":     "postsecondary teachers; all other",
    "writers and editors":                                "writers and authors",
    "criminal justice and law enforcement teachers postsecondary": "postsecondary teachers; all other",
    "environmental scientists and geoscientists":         "environmental scientists and specialists; including health",
    "clinical counseling and school psychologists":       "clinical; counseling; and school psychologists",
    "special education teachers":                         "special education teachers; all other",
    "special education teachers secondary school":        "special education teachers; secondary school",
    "exercise physiologists":                             "physical therapists",
    "area ethnic and cultural studies teachers postsecondary": "postsecondary teachers; all other",
    "information security analysts":                      "information security analysts",
    "architecture and engineering occupations":           "architects; except landscape and naval",
    "software developers systems software":               "software developers",
    "art and design workers":                             "art directors",
    "fine artists including painters sculptors and illustrators": "fine artists; including painters; sculptors; and illustrators",
    "nurse practitioners":                                "nurse practitioners",
    "nurse anesthetists":                                 "nurse anesthetists",
    "nurse midwives":                                     "nurse midwives",
    "nurse midwives and nurse anesthetists":              "nurse anesthetists",
    "designers":                                          "graphic designers",
    "chemists and materials scientists":                  "chemists",
    "other teachers and instructors":                     "postsecondary teachers; all other",
    "elementary and middle school teachers":              "elementary school teachers; except special education",
    "financial specialists":                              "financial analysts",
    "counselors all other":                               "counselors; all other",
    "conservation scientists and foresters":              "conservation scientists",
    "religious workers":                                  "clergy",
    "communications equipment operators":                 "telephone operators",
    "mathematical science teachers postsecondary":        "postsecondary teachers; all other",
    "computer network architects":                        "computer network architects",
    "biological science teachers postsecondary":          "postsecondary teachers; all other",
    "genetic counselors":                                 "genetic counselors",
    "actors producers and directors":                     "producers and directors",
    "agricultural and food scientists":                   "food scientists and technologists",
    "agents and business managers of artists performers and athletes": "agents and business managers of artists; performers; and athletes",
    "administrative law judges adjudicators and hearing officers": "administrative law judges; adjudicators; and hearing officers",
    "advertising marketing promotions public relations and sales managers": "marketing managers",
    "agricultural equipment operators":                   "agricultural workers; all other",
    "agricultural workers":                               "agricultural workers; all other",
    "agricultural workers all other":                     "agricultural workers; all other",
    "air traffic controllers and airfield operations specialists": "air traffic controllers",
    "air transportation workers":                         "airline pilots; copilots; and flight engineers",
    "aircraft pilots and flight engineers":               "airline pilots; copilots; and flight engineers",
    "aircraft structure surfaces rigging and systems assemblers": "aircraft structure; surfaces; rigging; and systems assemblers",
    "anesthesiologists":                                  "anesthesiologists and anesthesiologist assistants",
    "animal care and service workers":                    "animal care and service workers; all other",
    "arbitrators mediators and conciliators":             "arbitrators; mediators; and conciliators",
    "architects except landscape and naval":              "architects; except landscape and naval",
    "architects except naval":                            "architects; except landscape and naval",
    "architects surveyors and cartographers":             "architects; except landscape and naval",
    "preschool and kindergarten teachers":                "preschool teachers; except special education",
    "psychology teachers postsecondary":                  "postsecondary teachers; all other",
    "sales and related occupations":                      "sales representatives; wholesale and manufacturing",
    "secondary school teachers":                          "secondary school teachers; except special and career/technical education",
    "arts design entertainment sports and media occupations": "art directors",
    "personal care and service occupations":              "personal care aides",
    "physical scientists":                                "physical scientists; all other",
}

def resolve(name):
    if pd.isna(name): return name
    return full_map.get(str(name).lower().strip(), str(name).lower().strip())

bm = best_match.copy()
bm["career_key"] = bm["career_match"].apply(resolve)

gcsi_lookup = occupation_gcsi.copy()
gcsi_lookup["lookup_key"] = gcsi_lookup["occupation_name"].str.lower().str.strip()

base = bm.merge(
    gcsi_lookup[["lookup_key", "SOC6", "GCSI",
                 "automation_probability", "A_MEDIAN", "TOT_EMP"]],
    left_on="career_key",
    right_on="lookup_key",
    how="left"
)

gcsi_median = base["GCSI"].median()
base["GCSI_imputed"] = base["GCSI"].isna()
base["GCSI"]         = base["GCSI"].fillna(gcsi_median)
base["SOC6"]         = base["SOC6"].astype(str).str.strip()
base.drop(columns=["lookup_key", "career_key"], inplace=True)

# Skills
top_skills = (
    skill_bridge
    .dropna(subset=["SOC6", "skill_name", "importance"])
    .sort_values("importance", ascending=False)
    .groupby("SOC6").head(5)
    .groupby("SOC6")["skill_name"].apply(list)
    .reset_index()
    .rename(columns={"skill_name": "top_skills"})
)
top_skills["SOC6"] = top_skills["SOC6"].astype(str).str.strip()
top_tech = tech_summary[["SOC6", "technology_items", "hot_tech_items"]].copy()
top_tech["SOC6"] = top_tech["SOC6"].astype(str).str.strip()

print("Base enrichment done.")
print(f"  base shape: {base.shape}")
print(f"  GCSI median: {gcsi_median:.2f}")

Base enrichment done.
  base shape: (57085, 14)
  GCSI median: 41.78


In [3]:
def compare_weights(degree_name, top_n=8):
    """
    Shows recommendations for a degree under three weight schemes side by side.
    """
    mask = base["degree_final"].str.lower().str.contains(
        degree_name.lower().strip(), na=False
    )
    results = base[mask].copy()

    if results.empty:
        print(f"No matches for '{degree_name}'")
        return

    weights = {
        "50/50 (current)": (0.50, 0.50),
        "40/60 (GCSI leads)": (0.40, 0.60),
        "30/70 (GCSI dominates)": (0.30, 0.70),
    }

    print(f"\n{'='*70}")
    print(f"  DEGREE: {degree_name.upper()}")
    print(f"{'='*70}")

    for label, (w_sim, w_gcsi) in weights.items():
        df = results.copy()
        df["hybrid_score"] = (
            w_sim   * df["similarity"] +
            w_gcsi  * (df["GCSI"] / 100)
        ).round(4)

        top = (
            df.sort_values("hybrid_score", ascending=False)
            .drop_duplicates(subset=["career_match"])
            .head(top_n)
            .merge(top_skills, on="SOC6", how="left")
        )

        print(f"\n--- {label} ---")
        print(f"{'Rank':<5} {'Career':<45} {'Sim':>6} {'GCSI':>6} {'Hybrid':>8} {'Salary':>10}")
        print("-" * 85)
        for i, row in top.iterrows():
            salary_str = f"${row['A_MEDIAN']:,.0f}" if pd.notna(row["A_MEDIAN"]) else "N/A"
            imputed    = "*" if row["GCSI_imputed"] else " "
            print(f"{top.index.get_loc(i)+1:<5} {str(row['career_match']):<45} "
                  f"{row['similarity']:>6.3f} {row['GCSI']:>5.1f}{imputed} "
                  f"{row['hybrid_score']:>8.4f} {salary_str:>10}")

    print("\n* = GCSI imputed (median fallback, no exact salary data)")

In [4]:
test_degrees = [
    "computer science",
    "economics",
    "business administration",
    "nursing",
    "public administration",
    "data science",
]

for degree in test_degrees:
    compare_weights(degree, top_n=6)
    print()


  DEGREE: COMPUTER SCIENCE

--- 50/50 (current) ---
Rank  Career                                           Sim   GCSI   Hybrid     Salary
-------------------------------------------------------------------------------------
1     computer and information systems managers      0.691  61.9    0.6548   $135,800
2     electrical and electronics engineers           0.852  45.0    0.6512    $94,210
3     computer and information research scientists   0.769  47.7    0.6234   $111,840
4     computer hardware engineers                    0.798  44.5    0.6216   $115,080
5     computer systems analysts                      0.728  48.1    0.6044    $87,220
6     chemical engineers                             0.746  45.6    0.6014    $98,340

--- 40/60 (GCSI leads) ---
Rank  Career                                           Sim   GCSI   Hybrid     Salary
-------------------------------------------------------------------------------------
1     computer and information systems managers      0.691 

In [5]:
def compare_floors(degree_name, top_n=6):
    """
    Fixed at 40/60 weight, tests different similarity floors.
    """
    mask = base["degree_final"].str.lower().str.contains(
        degree_name.lower().strip(), na=False
    )
    results = base[mask].copy()

    if results.empty:
        print(f"No matches for '{degree_name}'")
        return

    floors = {
        "no floor (current)": 0.0,
        "floor 0.45":         0.45,
        "floor 0.50":         0.50,
        "floor 0.55":         0.55,
    }

    print(f"\n{'='*70}")
    print(f"  DEGREE: {degree_name.upper()}  |  weights: 40/60")
    print(f"{'='*70}")

    for label, floor in floors.items():
        df = results[results["similarity"] >= floor].copy()

        if df.empty:
            print(f"\n--- {label} --- NO RESULTS ABOVE FLOOR")
            continue

        df["hybrid_score"] = (
            0.40 * df["similarity"] +
            0.60 * (df["GCSI"] / 100)
        ).round(4)

        top = (
            df.sort_values("hybrid_score", ascending=False)
            .drop_duplicates(subset=["career_match"])
            .head(top_n)
        )

        total_candidates = len(df["career_match"].unique())

        print(f"\n--- {label}  ({total_candidates} candidates above floor) ---")
        print(f"{'Rank':<5} {'Career':<45} {'Sim':>6} {'GCSI':>6} {'Hybrid':>8} {'Salary':>10}")
        print("-" * 85)
        for rank, (_, row) in enumerate(top.iterrows(), start=1):
            salary_str = f"${row['A_MEDIAN']:,.0f}" if pd.notna(row["A_MEDIAN"]) else "N/A"
            imputed    = "*" if row["GCSI_imputed"] else " "
            print(f"{rank:<5} {str(row['career_match']):<45} "
                  f"{row['similarity']:>6.3f} {row['GCSI']:>5.1f}{imputed} "
                  f"{row['hybrid_score']:>8.4f} {salary_str:>10}")

    print("\n* = GCSI imputed (median fallback)")


# Test the problem degrees only
for degree in ["nursing", "economics", "public administration", "data science"]:
    compare_floors(degree, top_n=6)
    print()


  DEGREE: NURSING  |  weights: 40/60

--- no floor (current)  (23 candidates above floor) ---
Rank  Career                                           Sim   GCSI   Hybrid     Salary
-------------------------------------------------------------------------------------
1     registered nurses                              0.926  60.2    0.7319    $68,450
2     chief executives                               0.551  71.4    0.6487   $181,210
3     nurse anesthetists                             0.948  41.8*   0.6300        N/A
4     nurse practitioners                            0.920  41.8*   0.6187        N/A
5     nurse midwives                                 0.917  41.8*   0.6175        N/A
6     nursing assistants                             0.781  41.8*   0.5631        N/A

--- floor 0.45  (23 candidates above floor) ---
Rank  Career                                           Sim   GCSI   Hybrid     Salary
----------------------------------------------------------------------------------

In [6]:
def compare_penalty(degree_name, top_n=6):
    """
    Tests a similarity-weighted GCSI penalty.
    Low similarity careers get their GCSI contribution reduced proportionally.
    Formula: hybrid = sim_weight * sim + gcsi_weight * (GCSI/100 * sim_boost)
    where sim_boost = similarity / 0.7 (careers above 0.7 sim get full GCSI credit)
    """
    mask = base["degree_final"].str.lower().str.contains(
        degree_name.lower().strip(), na=False
    )
    results = base[mask].copy()

    if results.empty:
        print(f"No matches for '{degree_name}'")
        return

    approaches = {
        "40/60 no penalty (current best)": 
            lambda df: 0.40 * df["similarity"] + 0.60 * (df["GCSI"] / 100),
        
        "40/60 + sim-scaled GCSI (boost threshold 0.65)":
            lambda df: 0.40 * df["similarity"] + 0.60 * (df["GCSI"] / 100) * (df["similarity"] / 0.65).clip(upper=1.0),
        
        "40/60 + sim-scaled GCSI (boost threshold 0.55)":
            lambda df: 0.40 * df["similarity"] + 0.60 * (df["GCSI"] / 100) * (df["similarity"] / 0.55).clip(upper=1.0),

        "40/60 + sim-scaled GCSI (boost threshold 0.70)":
            lambda df: 0.40 * df["similarity"] + 0.60 * (df["GCSI"] / 100) * (df["similarity"] / 0.70).clip(upper=1.0),
    }

    print(f"\n{'='*70}")
    print(f"  DEGREE: {degree_name.upper()}")
    print(f"{'='*70}")

    for label, score_fn in approaches.items():
        df = results.copy()
        df["hybrid_score"] = score_fn(df).round(4)

        top = (
            df.sort_values("hybrid_score", ascending=False)
            .drop_duplicates(subset=["career_match"])
            .head(top_n)
        )

        print(f"\n--- {label} ---")
        print(f"{'Rank':<5} {'Career':<45} {'Sim':>6} {'GCSI':>6} {'Hybrid':>8} {'Salary':>10}")
        print("-" * 85)
        for rank, (_, row) in enumerate(top.iterrows(), start=1):
            salary_str = f"${row['A_MEDIAN']:,.0f}" if pd.notna(row["A_MEDIAN"]) else "N/A"
            imputed    = "*" if row["GCSI_imputed"] else " "
            print(f"{rank:<5} {str(row['career_match']):<45} "
                  f"{row['similarity']:>6.3f} {row['GCSI']:>5.1f}{imputed} "
                  f"{row['hybrid_score']:>8.4f} {salary_str:>10}")

    print("\n* = GCSI imputed (median fallback)")


# Focus on the problem degrees
for degree in ["nursing", "economics", "public administration", "business administration"]:
    compare_penalty(degree, top_n=6)
    print()


  DEGREE: NURSING

--- 40/60 no penalty (current best) ---
Rank  Career                                           Sim   GCSI   Hybrid     Salary
-------------------------------------------------------------------------------------
1     registered nurses                              0.926  60.2    0.7319    $68,450
2     chief executives                               0.551  71.4    0.6487   $181,210
3     nurse anesthetists                             0.948  41.8*   0.6300        N/A
4     nurse practitioners                            0.920  41.8*   0.6187        N/A
5     nurse midwives                                 0.917  41.8*   0.6175        N/A
6     nursing assistants                             0.781  41.8*   0.5631        N/A

--- 40/60 + sim-scaled GCSI (boost threshold 0.65) ---
Rank  Career                                           Sim   GCSI   Hybrid     Salary
-------------------------------------------------------------------------------------
1     registered nurses 

In [3]:
import pandas as pd
import os

DATASETS = "datasets"

files = [
    ("occupation_gcsi.csv",                           "occupation_gcsi.parquet"),
    ("program_recommendations.csv",                   "program_recommendations.parquet"),
    ("program_field_dropdown.csv",                    "program_field_dropdown.parquet"),
    ("occupation_skill_bridge_onet.csv",              "occupation_skill_bridge_onet.parquet"),
    ("occupation_technology_skills_onet_summary.csv", "occupation_technology_skills_onet_summary.parquet"),
]

print("Converting CSVs to Parquet...")
for csv_name, parquet_name in files:
    csv_path     = f"{DATASETS}/{csv_name}"
    parquet_path = f"{DATASETS}/{parquet_name}"

    df = pd.read_csv(csv_path, low_memory=False)
    df.to_parquet(parquet_path, engine="fastparquet", index=False)

    csv_size     = os.path.getsize(csv_path) / (1024*1024)
    parquet_size = os.path.getsize(parquet_path) / (1024*1024)
    reduction    = (1 - parquet_size/csv_size) * 100

    print(f"  {csv_name:<55} {csv_size:>7.1f} MB → {parquet_size:>6.1f} MB  ({reduction:.0f}% smaller)")

total = sum(
    os.path.getsize(f"{DATASETS}/{p}")
    for _, p in files
) / (1024*1024)
print(f"\nTotal parquet: {total:.1f} MB")

Converting CSVs to Parquet...
  occupation_gcsi.csv                                         0.1 MB →    0.1 MB  (51% smaller)
  program_recommendations.csv                               103.9 MB →   33.5 MB  (68% smaller)
  program_field_dropdown.csv                                  2.4 MB →    0.7 MB  (70% smaller)
  occupation_skill_bridge_onet.csv                           12.6 MB →    1.6 MB  (87% smaller)
  occupation_technology_skills_onet_summary.csv               0.1 MB →    0.0 MB  (42% smaller)

Total parquet: 35.9 MB
